# Bisherige Methodenvergleiche

Dieses Notebook enthält die bisherigen Vergleiche der Stopwörter, der Anzahl von Topics, der NMF-Skalierung und der Ergänzung `leuchte → beleuchtung`.
Die damalige Konfiguration bleibt erhalten, damit die Versuche unabhängig von der weiterentwickelten Analyse nachvollziehbar sind.

Die aktuelle Analyse steht in [01_themenanalyse.ipynb](01_themenanalyse.ipynb). Dieses Notebook muss dafür nicht ausgeführt werden.
Die Zellen hier sind in einer eigenen Sitzung von oben nach unten auszuführen. Ergebnisse werden unter `outputs/bisherige_vergleiche/` gespeichert.

Die Untersuchung einzelner Meldungen zeigte auch unpassende Zuordnungen. Die alten Themenbezeichnungen bleiben deshalb deaktiviert.

Die ursprünglichen Vergleiche wurden mit der Originaldatei durchgeführt. Bei einer erneuten Ausführung wird die mitgelieferte bereinigte CSV verwendet. Einzelne Ergebnisse können deshalb von den dokumentierten Werten abweichen. Die damaligen Modellparameter und Textregeln bleiben erhalten.


In [ ]:
# Schritt 01 Umgebung und Arbeitsordner prüfen

from pathlib import Path
import sys
import re

import pandas as pd
import numpy as np
import sklearn
from IPython.display import display

print("Python executable:")
print(sys.executable)

print("\nPandas Version:")
print(pd.__version__)

print("\nNumPy Version:")
print(np.__version__)

print("\nScikit Learn Version:")
print(sklearn.__version__)

print("\nAktueller Arbeitsordner:")
print(Path.cwd())


In [ ]:
# Schritt 02 Konfiguration

CSV_NAME = "maengelmelder_konstanz_final_9_bereinigt.csv"
TEXT_COLUMN = "text_raw"
ID_COLUMN = "id"

# Kurze Meldungen bleiben erhalten, sofern nach dem Preprocessing ein Token verbleibt.
MIN_WORDS_RAW = 1
MIN_TOKENS_CLEAN = 1
MIN_TOKEN_LENGTH = 2

# Parameter der Vektorisierung
MIN_DF = 5
MAX_DF = 0.90
NGRAM_RANGE = (1, 1)
TOKEN_PATTERN = r"(?u)\b[a-zäöüß][a-zäöüß]+\b"
SUBLINEAR_TF = True

# Bisherige Modelleinstellung, die Auswahl der Topic-Anzahl steht noch aus.
N_TOPICS = 10
RANDOM_STATE = 42
LDA_MAX_ITER = 20
NMF_MAX_ITER = 500

# Erst nach der inhaltlichen Prüfung der Bezeichnungen für das aktuelle Modell aktivieren.
TOPIC_LABELS_REVIEWED = False

# Deutsche Stopwörter aus dem bisherigen Prototyp
GERMAN_STOPWORDS = {
    "aber", "alle", "allem", "allen", "aller", "alles", "als", "also", "am", "an",
    "andere", "anderem", "anderen", "anderer", "anderes", "auch", "auf", "aus",
    "bei", "bin", "bis", "bist", "da", "damit", "dann", "der", "den", "des",
    "dem", "die", "das", "dass", "daß", "dazu", "dein", "deine", "deinem",
    "deinen", "deiner", "deines", "denn", "dich", "dir", "du", "dies", "diese",
    "diesem", "diesen", "dieser", "dieses", "doch", "dort", "durch", "ein",
    "eine", "einem", "einen", "einer", "eines", "einige", "einigen", "einiger",
    "einiges", "einmal", "er", "ihn", "ihm", "es", "etwas", "euer", "eure",
    "eurem", "euren", "eurer", "eures", "für", "gegen", "gewesen", "hab",
    "habe", "haben", "hat", "hatte", "hatten", "hier", "hin", "hinter", "ich",
    "mich", "mir", "ihr", "ihre", "ihrem", "ihren", "ihrer", "ihres", "euch",
    "im", "in", "indem", "ins", "ist", "jede", "jedem", "jeden", "jeder",
    "jedes", "jene", "jenem", "jenen", "jener", "jenes", "jetzt", "kann",
    "kein", "keine", "keinem", "keinen", "keiner", "keines", "können", "könnte",
    "machen", "man", "manche", "manchem", "manchen", "mancher", "manches",
    "mein", "meine", "meinem", "meinen", "meiner", "meines", "mit", "muss",
    "musste", "nach", "nicht", "nichts", "noch", "nun", "nur", "ob", "oder",
    "ohne", "sehr", "sein", "seine", "seinem", "seinen", "seiner", "seines",
    "selbst", "sich", "sie", "ihnen", "sind", "so", "solche", "solchem",
    "solchen", "solcher", "solches", "soll", "sollte", "sondern", "sonst",
    "über", "um", "und", "uns", "unser", "unsere", "unserem", "unseren",
    "unserer", "unseres", "unter", "viel", "vom", "von", "vor", "während",
    "war", "waren", "warst", "was", "weg", "weil", "weiter", "welche",
    "welchem", "welchen", "welcher", "welches", "wenn", "werde", "werden",
    "wie", "wieder", "will", "wir", "wird", "wirst", "wo", "wollen", "wollte",
    "würde", "würden", "zu", "zum", "zur", "zwar", "zwischen"
}

# Zusammengeführte Ergänzungen aus den bisherigen Versuchen.
DOMAIN_STOPWORDS = {
    "abends", "abgeholt", "abgelaufen", "abgelegt", "abgeschlossen", "abgestellt",
    "abgestellte", "abgestellter", "abgestelltes", "abholung", "adresssuche", "aktuell",
    "allerdings", "allmannsdorf", "allmannshoehe", "alte", "alter", "altes", "andere",
    "anderen", "ausgefallen", "autom", "beethovenstr", "befinden", "befindet", "beiden", "beim",
    "bereich", "bereits", "beschaedigt", "beschädigt", "bild", "bilder", "bitte", "brennen",
    "brennt", "ca", "circa", "dank", "danke", "dankeschoen", "dankeschön", "defekt", "defekte",
    "defekten", "defekter", "dienstag", "direkt", "donnerstag", "dorffriedhofsweg", "dunkel",
    "ecke", "entfernen", "entsorgt", "ermittelt", "etc", "flurweg", "foto", "fotos", "freitag",
    "frieda", "fuer", "funktion", "funktioniert", "fuss", "ganze", "gegenueber", "gegenüber",
    "gehen", "geht", "gemeldet", "gestern", "guten", "haengt", "hallo", "hans",
    "hansegartenstrasse", "haus", "hausnummer", "heugaessle", "heute", "hinten", "hoehe",
    "hoehenweg", "hängt", "höhe", "immer", "jahr", "jahren", "kaputt", "kaputte", "kaputtes",
    "kaum", "koennen", "koennte", "kommen", "kommt", "komplett", "konstanz", "konstanzer",
    "kreuzlinger", "können", "könnte", "laengerem", "leider", "leuchte", "leuchten", "leuchtet",
    "liegen", "liegt", "litzelstetten", "macairestrasse", "mainaustrasse", "mal", "mannheimer",
    "max", "mehr", "mehrere", "mehreren", "mehrfach", "mfg", "mindestens", "mittags",
    "mittwoch", "moeglich", "moegliche", "moeglicherweise", "monat", "monaten", "montag",
    "morgen", "möglich", "nacht", "naehe", "nahe", "neben", "nr", "nummer", "oben", "ort",
    "peter", "richtung", "samstag", "schaden", "schadhaft", "schaeden", "schneckenburgstrasse",
    "schoen", "schon", "schwach", "schön", "seit", "seite", "siehe", "sieht", "sigrist",
    "sonnentauweg", "sonntag", "stadt", "stark", "stehen", "steht", "steinstrasse", "stelle",
    "str", "stromeyer", "tag", "tage", "tagen", "teil", "teile", "thumb", "ueber", "unten",
    "untere", "unteren", "urisberg", "vermutlich", "viele", "vielen", "vorne", "waere",
    "waeren", "weiteres", "woche", "wochen", "worden", "wuerde", "wuerden", "wurde", "wurden",
    "wäre", "wären", "zeit", "zwei",
}

STOPWORDS_FINAL = GERMAN_STOPWORDS | DOMAIN_STOPWORDS

# Bisherige Zuordnung ausgewählter Schreibweisen und Begriffe
CANONICAL_TERMS_FINAL = {
    "abfaelle": "müll",
    "abfall": "müll",
    "abfälle": "müll",
    "aeste": "äste",
    "baeume": "baum",
    "beschaedigte": "beschädigt",
    "beschaedigten": "beschädigt",
    "beschädigte": "beschädigt",
    "bäume": "baum",
    "container": "müllcontainer",
    "fahrrads": "fahrrad",
    "fahrraeder": "fahrrad",
    "fahrräder": "fahrrad",
    "fuer": "für",
    "fussgaenger": "fußgänger",
    "fussgaengerampel": "ampel",
    "fussweg": "gehweg",
    "fusswege": "gehweg",
    "fußgängerampel": "ampel",
    "fußweg": "gehweg",
    "fußwege": "gehweg",
    "gefaehrlich": "gefährlich",
    "gehwege": "gehweg",
    "grobmuell": "grobmüll",
    "gruen": "grün",
    "gullideckel": "gulli",
    "gullis": "gulli",
    "haengt": "hängt",
    "hecken": "hecke",
    "heckenwuchs": "hecke",
    "koennen": "können",
    "laeuft": "läuft",
    "lampe": "beleuchtung",
    "lampen": "beleuchtung",
    "laterne": "beleuchtung",
    "laternen": "beleuchtung",
    "moeglich": "möglich",
    "muell": "müll",
    "muelleimer": "mülleimer",
    "müllcontainer": "müll",
    "mülltonne": "müll",
    "mülltonnen": "müll",
    "radwege": "radweg",
    "schilder": "schild",
    "straeucher": "sträucher",
    "strasse": "straße",
    "strassen": "straße",
    "strassenbeleuchtung": "beleuchtung",
    "strassenlampe": "beleuchtung",
    "strassenlampen": "beleuchtung",
    "strassenlaterne": "beleuchtung",
    "strassenlaternen": "beleuchtung",
    "strassenleuchte": "beleuchtung",
    "strassenleuchten": "beleuchtung",
    "strassenseiten": "straßenseiten",
    "strassensteiten": "straßenseiten",
    "strauch": "sträucher",
    "straßenbeleuchtung": "beleuchtung",
    "straßenlaterne": "beleuchtung",
    "straßenlaternen": "beleuchtung",
    "ueber": "über",
    "ueberfuellt": "überfüllt",
    "umgestuerzt": "umgestürzt",
    "verkehrsschild": "schild",
    "verkehrsschilder": "schild",
    "verschmutzte": "verschmutzt",
    "verschmutzten": "verschmutzt",
    "verunreinigt": "verschmutzt",
    "waechst": "wächst",
    "waere": "wäre",
    "zerstoert": "zerstört",
}


In [ ]:
# Schritt 03 CSV-Datei laden

from pathlib import Path
import pandas as pd

possible_paths = [
    Path("data") / "raw" / CSV_NAME,
    Path("..") / "data" / "raw" / CSV_NAME,
]

csv_path = None

for path in possible_paths:
    print("Prüfe:", path.resolve())
    if path.exists():
        csv_path = path
        break

if csv_path is None:
    raise FileNotFoundError(
        "CSV wurde nicht gefunden. Prüfe, ob die Datei im Ordner data/raw liegt."
    )

print("\nGefundene CSV:")
print(csv_path.resolve())

# Der Projektordner ergibt sich aus dem Pfad data/raw der gefundenen CSV.
PROJECT_DIR = csv_path.parent.resolve().parent.parent

df = pd.read_csv(csv_path)

print("\nShape:")
print(df.shape)

print("\nSpalten:")
print(list(df.columns))

df.head(3)


In [ ]:
# Schritt 04 Data Profiling

required_columns = [
    "id",
    "date_created",
    TEXT_COLUMN,
    "category_parent",
    "category_child",
    "responsible",
    "lat",
    "lon",
    "address_input",
    "detail_url",
]

missing_required = [col for col in required_columns if col not in df.columns]

if missing_required:
    raise ValueError(f"Folgende erwartete Spalten fehlen: {missing_required}")

print("Anzahl Zeilen:", len(df))

print("\nFehlende Werte je Spalte:")
print(df.isna().sum())

print("\nFehlende Werte in text_raw:")
print(df[TEXT_COLUMN].isna().sum())

print("\nDubletten in text_raw:")
print(df[TEXT_COLUMN].duplicated().sum())

print("\nKategorien in category_parent:")
print(df["category_parent"].value_counts(dropna=False))

print("\nKategorien in category_child:")
print(df["category_child"].value_counts(dropna=False).head(25))

print("\nTextlängen in Wörtern:")
word_counts_raw = df[TEXT_COLUMN].dropna().astype(str).str.split().str.len()
print(word_counts_raw.describe())

print("\nBeispiele aus text_raw:")
for i, text in enumerate(df[TEXT_COLUMN].dropna().astype(str).head(5), start=1):
    print(f"\nBeispiel {i}:")
    print(text[:500])


In [ ]:
# Schritt 05 Arbeitsdatensatz erstellen

def prepare_records(
    source_df: pd.DataFrame,
    text_column: str,
    min_words: int,
    id_column: str,
):
    """Doppelte Datensätze entfernen und Meldungen mit unterschiedlichen IDs beibehalten."""
    if not source_df.columns.is_unique:
        raise ValueError("Die Spaltennamen müssen eindeutig sein.")
    missing_columns = [col for col in (text_column, id_column) if col not in source_df.columns]
    if missing_columns:
        raise ValueError(f"Erforderliche Spalten fehlen: {missing_columns}")
    if text_column == id_column:
        raise ValueError("Textspalte und ID-Spalte müssen unterschiedlich sein.")
    if not isinstance(min_words, int) or isinstance(min_words, bool) or min_words < 1:
        raise ValueError("min_words muss eine ganze Zahl größer oder gleich 1 sein.")

    counts = {"input": len(source_df)}

    # Fehlende IDs nicht als gemeinsame Meldung behandeln.
    id_keys = source_df[id_column].astype("string").str.strip()
    if id_keys.isna().any() or id_keys.eq("").any():
        raise ValueError("Meldungs-IDs fehlen. Die betroffenen Datensätze müssen geprüft werden.")

    # Nur über alle Spalten identische Datensätze entfernen, einschließlich der ID.
    work = source_df.drop_duplicates(keep="first").copy()
    counts["duplicates"] = len(source_df) - len(work)

    # Bei unterschiedlichen Inhalten zur selben ID keinen Datensatz willkürlich auswählen.
    if work[id_column].astype("string").str.strip().duplicated(keep=False).any():
        raise ValueError(
            "Zu einer Meldungs-ID liegen unterschiedliche Datensätze vor. "
            "Die Unterschiede müssen vor der Analyse geprüft werden."
        )

    before = len(work)
    work = work.dropna(subset=[text_column]).copy()
    counts["missing"] = before - len(work)
    work[text_column] = work[text_column].astype(str).str.strip()

    before = len(work)
    work = work.loc[work[text_column].str.len() > 0].copy()
    counts["empty"] = before - len(work)

    # Gleiche Texte mit unterschiedlichen IDs zählen weiterhin als einzelne Meldungen.
    work["word_count_raw"] = work[text_column].str.split().str.len()
    before = len(work)
    work = work.loc[work["word_count_raw"] >= min_words].copy()
    counts["raw_short"] = before - len(work)
    counts["remaining"] = len(work)
    counts["text_repetitions"] = int(work[text_column].duplicated().sum())

    if counts["input"] != sum(counts[key] for key in (
        "duplicates", "missing", "empty", "raw_short", "remaining"
    )):
        raise RuntimeError("Die dokumentierten Zeilenzahlen sind nicht konsistent.")

    return work.reset_index(drop=True), counts


df_work, record_counts = prepare_records(df, TEXT_COLUMN, MIN_WORDS_RAW, ID_COLUMN)

print("Ausgangszeilen:", record_counts["input"])
print("Entfernte vollständig identische Datensätze:", record_counts["duplicates"])
print("Entfernte fehlende Texte:", record_counts["missing"])
print("Entfernte leere Texte:", record_counts["empty"])
print("Entfernte Texte unter der Mindestlänge:", record_counts["raw_short"])
print("Meldungen im Arbeitsdatensatz:", record_counts["remaining"])
print("Weitere Meldungen mit gleichem Text, beibehalten:", record_counts["text_repetitions"])

# Unterschiedliche IDs mit gleichem Text bleiben erhalten, eine identische Zeile entfällt.
_test_data = pd.DataFrame({
    ID_COLUMN: [1, 2, 3, 4, 5, 6, 7, 8, 8],
    TEXT_COLUMN: [None, np.nan, pd.NA, "", "  ", " Müll ", "Müll", "Lampe defekt", "Lampe defekt"],
})
_test_work, _test_counts = prepare_records(_test_data, TEXT_COLUMN, 1, ID_COLUMN)
assert _test_work[ID_COLUMN].tolist() == [6, 7, 8]
assert _test_work[TEXT_COLUMN].tolist() == ["Müll", "Müll", "Lampe defekt"]
assert _test_counts == {
    "input": 9, "duplicates": 1, "missing": 3, "empty": 2,
    "raw_short": 0, "remaining": 3, "text_repetitions": 1,
}
print("Funktionstest für Fehlwerte und Dubletten: bestanden")

df_work[[ID_COLUMN, TEXT_COLUMN, "category_parent", "category_child", "word_count_raw"]].head(10)

In [ ]:
# Schritt 06 Ausgabeordner festlegen

OUTPUT_DIR = PROJECT_DIR / "outputs" / "bisherige_vergleiche"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output-Ordner:")
print(OUTPUT_DIR)


In [ ]:
# Schritt 07 Text Preprocessing definieren

# Die Endungsregeln erkennen mögliche Ortsangaben, aber nicht alle Ortsnamen.
def is_location_token(token: str) -> bool:
    # Mögliche Straßennamen anhand der Endung erkennen und "straße" ausnehmen.
    if token == "straße":
        return False

    if token.endswith(("strasse", "straße", "str")) and len(token) > 5:
        return True

    # Mögliche Wegnamen erkennen und die folgenden allgemeinen Wegbegriffe ausnehmen.
    if token in {"gehweg", "radweg", "fahrradweg"}:
        return False

    if token.endswith("weg") and len(token) > 5:
        return True

    return False

def clean_text(text: str) -> str:
    text = str(text).lower()

    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-zäöüß\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = []

    for token in text.split():
        token = CANONICAL_TERMS_FINAL.get(token, token)

        if is_location_token(token):
            continue

        if token not in STOPWORDS_FINAL and len(token) >= MIN_TOKEN_LENGTH:
            tokens.append(token)

    return " ".join(tokens)


In [ ]:
# Schritt 08 Text Preprocessing anwenden

# Für jeden Lauf eine neue Kopie des Arbeitsdatensatzes erzeugen.
df_model = df_work.copy()
df_model["text_clean"] = df_model[TEXT_COLUMN].apply(clean_text)
df_model["word_count_clean"] = df_model["text_clean"].str.split().str.len()

before_clean_filter = len(df_model)
df_model = df_model[df_model["word_count_clean"] >= MIN_TOKENS_CLEAN].copy()
after_clean_filter = len(df_model)
df_model = df_model.reset_index(drop=True)

print("Zeilen vor Textfilter:", before_clean_filter)
print("Zeilen nach Textfilter:", after_clean_filter)
print("Entfernte Meldungen:", before_clean_filter - after_clean_filter)
print("\nVerteilung der Wortanzahl nach Text Preprocessing:")
print(df_model["word_count_clean"].describe())


In [ ]:
# Schritt 09 Bereinigte Texte anhand von Beispielen ansehen

print("Beispiele vorher/nachher:")

for _, row in df_model[[TEXT_COLUMN, "text_clean"]].head(10).iterrows():
    print("\nRAW:")
    print(row[TEXT_COLUMN][:300])
    print("CLEAN:")
    print(row["text_clean"])


In [ ]:
# Schritt 10 Texte mit CountVectorizer und TfidfVectorizer vektorisieren

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

texts_clean = df_model["text_clean"].tolist()

count_vectorizer = CountVectorizer(
    min_df=MIN_DF,
    max_df=MAX_DF,
    token_pattern=TOKEN_PATTERN,
    ngram_range=NGRAM_RANGE
)

tfidf_vectorizer = TfidfVectorizer(
    min_df=MIN_DF,
    max_df=MAX_DF,
    token_pattern=TOKEN_PATTERN,
    ngram_range=NGRAM_RANGE,
    sublinear_tf=SUBLINEAR_TF
)

X_count = count_vectorizer.fit_transform(texts_clean)
X_tfidf = tfidf_vectorizer.fit_transform(texts_clean)

print("Anzahl Dokumente:", len(texts_clean))
print("Count-Matrix:", X_count.shape)
print("TF-IDF-Matrix:", X_tfidf.shape)

print("\nBeispiele aus Count-Vokabular:")
print(count_vectorizer.get_feature_names_out()[:30])

print("\nBeispiele aus TF-IDF-Vokabular:")
print(tfidf_vectorizer.get_feature_names_out()[:30])


In [ ]:
# Kontrolle vor der Modellierung

print("Eingelesene Meldungen:", len(df))
print("Meldungen nach Text Preprocessing:", len(df_model))
print("Count-Matrix:", X_count.shape)
print("TF-IDF-Matrix:", X_tfidf.shape)
print(
    "Meldungen ohne Merkmale in der Count-Matrix:",
    int((X_count.getnnz(axis=1) == 0).sum())
)
print(
    "Meldungen ohne Merkmale in der TF-IDF-Matrix:",
    int((X_tfidf.getnnz(axis=1) == 0).sum())
)

# Kontrollieren, dass nach dem Preprocessing keine leeren Texte verbleiben.
assert df_model[TEXT_COLUMN].notna().all()
assert (df_model["word_count_clean"] >= MIN_TOKENS_CLEAN).all()
assert len(df_model) == X_count.shape[0] == X_tfidf.shape[0]

# Kurze, inhaltliche Testmeldungen bleiben mit der neuen Mindestlänge erhalten.
assert len(clean_text("Lampe defekt").split()) >= MIN_TOKENS_CLEAN
assert len(clean_text("Müll").split()) >= MIN_TOKENS_CLEAN
assert clean_text("Bitte danke") == ""
print("Kontrolle der Textfilter: bestanden")



In [ ]:
# Schritt 10b Meldungen ohne Merkmale erfassen

def select_model_documents(documents, count_matrix, tfidf_matrix):
    """Die gleiche Auswahl von Meldungen auf beide Merkmalsmatrizen anwenden."""
    if len(documents) != count_matrix.shape[0] or count_matrix.shape != tfidf_matrix.shape:
        raise ValueError("Meldungen und Merkmalsmatrizen passen nicht zusammen.")
    if not documents.index.is_unique:
        raise ValueError("Der Index der Meldungen muss eindeutig sein.")

    count_matrix = count_matrix.tocsr(copy=True)
    tfidf_matrix = tfidf_matrix.tocsr(copy=True)
    for matrix in (count_matrix, tfidf_matrix):
        if not np.isfinite(matrix.data).all() or (matrix.data < 0).any():
            raise ValueError("Merkmalsmatrizen müssen endliche, nichtnegative Werte enthalten.")
        matrix.eliminate_zeros()

    count_mask = np.asarray(count_matrix.getnnz(axis=1)).ravel() > 0
    tfidf_mask = np.asarray(tfidf_matrix.getnnz(axis=1)).ravel() > 0
    if not np.array_equal(count_mask, tfidf_mask):
        raise ValueError("Count und TF-IDF ergeben unterschiedliche Meldungen ohne Merkmale.")

    # Die ursprünglichen Zeilenpositionen bleiben über den Index nachvollziehbar.
    fit_documents = documents.iloc[np.flatnonzero(count_mask)].copy()
    without_features = documents.iloc[np.flatnonzero(~count_mask)].copy()
    return (
        count_mask,
        fit_documents,
        count_matrix[count_mask],
        tfidf_matrix[count_mask],
        without_features,
    )


if not np.array_equal(
    count_vectorizer.get_feature_names_out(), tfidf_vectorizer.get_feature_names_out()
):
    raise ValueError("Count und TF-IDF verwenden unterschiedliche Vokabulare.")

model_mask, df_fit, X_count_fit, X_tfidf_fit, df_without_features = select_model_documents(
    df_model, X_count, X_tfidf
)
N_ANALYSIS_DOCS = len(df_model)
if N_ANALYSIS_DOCS == 0:
    raise ValueError("Nach dem Text Preprocessing sind keine Meldungen vorhanden.")

# Vokabular und IDF bleiben aus dem gesamten Korpus nach Text Preprocessing erhalten.
# Nur für das Training werden die Zeilen ohne Merkmale ausgeschlossen.
assert len(df_fit) + len(df_without_features) == N_ANALYSIS_DOCS
assert len(df_fit) == X_count_fit.shape[0] == X_tfidf_fit.shape[0]
assert (X_count_fit.getnnz(axis=1) > 0).all()
assert (X_tfidf_fit.getnnz(axis=1) > 0).all()

print("Meldungen nach Text Preprocessing:", N_ANALYSIS_DOCS)
print("Meldungen mit Merkmalen für das Modelltraining:", len(df_fit))
print("Nicht zugeordnet, da keine Merkmale im Vokabular:", len(df_without_features))
print("Count-Matrix für das Modelltraining:", X_count_fit.shape)
print("TF-IDF-Matrix für das Modelltraining:", X_tfidf_fit.shape)
print("Meldungen ohne Merkmale in den Trainingsmatrizen:", 0)
print("Bezugsgröße der Prozentanteile:", N_ANALYSIS_DOCS)
print("Kontrolle der Auswahl: bestanden")


In [ ]:
# Schritt 11 LDA und NMF trainieren

from sklearn.decomposition import LatentDirichletAllocation, NMF

if len(df_fit) == 0:
    raise ValueError("Keine Meldungen mit Merkmalen vorhanden. Daten und Vokabularfilter prüfen.")
if not isinstance(N_TOPICS, int) or isinstance(N_TOPICS, bool) or N_TOPICS < 1:
    raise ValueError("N_TOPICS muss eine positive ganze Zahl sein.")
if N_TOPICS > min(X_tfidf_fit.shape):
    raise ValueError("Für diese NMF-Initialisierung gibt es zu wenige Meldungen oder Merkmale.")

lda_model = LatentDirichletAllocation(
    n_components=N_TOPICS,
    random_state=RANDOM_STATE,
    learning_method="batch",
    max_iter=LDA_MAX_ITER
)

nmf_model = NMF(
    n_components=N_TOPICS,
    random_state=RANDOM_STATE,
    init="nndsvda",
    max_iter=NMF_MAX_ITER
)

lda_doc_topic = lda_model.fit_transform(X_count_fit)
nmf_doc_topic = nmf_model.fit_transform(X_tfidf_fit)

print("LDA Dokument-Topic-Matrix:", lda_doc_topic.shape)
print("NMF Dokument-Topic-Matrix:", nmf_doc_topic.shape)


In [ ]:
# Schritt 11b Topic-Zuordnung und nicht zugeordnete Meldungen erfassen

def assign_topics(doc_topic_matrix, model_mask, source_index):
    """Modellgewichte den ursprünglichen Zeilen zuordnen und fehlende Merkmale kennzeichnen."""
    weights = np.asarray(doc_topic_matrix, dtype=float)
    mask = np.asarray(model_mask)
    if mask.ndim != 1 or mask.dtype != bool or len(mask) != len(source_index):
        raise ValueError("Die Zeilenauswahl passt nicht zum Index der Meldungen.")
    if not source_index.is_unique:
        raise ValueError("Der Index der Meldungen muss eindeutig sein.")
    if weights.ndim != 2 or weights.shape[0] != int(mask.sum()) or weights.shape[1] == 0:
        raise ValueError("Die Dokument-Topic-Matrix passt nicht zur Zeilenauswahl.")
    if not np.isfinite(weights).all() or (weights < 0).any():
        raise ValueError("Topic-Gewichte müssen endlich und nichtnegativ sein.")

    result = pd.DataFrame(index=source_index.copy())
    result["Topic"] = pd.Series(pd.NA, index=source_index, dtype="Int64")
    result["Status"] = pd.Series(
        "Keine Merkmale im Vokabular", index=source_index, dtype="string"
    )
    result["Gleicher Höchstwert"] = False
    positions = np.flatnonzero(mask)
    if len(positions) == 0:
        return result

    max_weights = weights.max(axis=1)
    positive = max_weights > 0
    valid_index = source_index.take(positions[positive])
    zero_index = source_index.take(positions[~positive])

    # Nur positive Gewichte für eine Zuordnung verwenden.
    # Bei exakt gleichen Höchstwerten wählt argmax die erste Topic-Nummer.
    labels = weights[positive].argmax(axis=1) + 1
    result.loc[valid_index, "Topic"] = labels
    result.loc[valid_index, "Status"] = "Topic zugeordnet"
    result.loc[zero_index, "Status"] = "Keine positiven Topic-Gewichte"
    tied = (weights[positive] == max_weights[positive, None]).sum(axis=1) > 1
    result.loc[valid_index, "Gleicher Höchstwert"] = tied
    return result


lda_assignments = assign_topics(lda_doc_topic, model_mask, df_model.index)
nmf_assignments = assign_topics(nmf_doc_topic, model_mask, df_model.index)

# Künstlicher Test: eine zugeordnete Meldung, eine ohne Merkmale und eine ohne positive Gewichte.
_test_assignments = assign_topics(
    np.array([[0.2, 0.8], [0.0, 0.0]]),
    np.array([True, False, True]),
    pd.Index([10, 20, 30]),
)
assert _test_assignments.loc[10, "Topic"] == 2
assert _test_assignments.loc[[20, 30], "Topic"].isna().all()
assert _test_assignments.loc[20, "Status"] == "Keine Merkmale im Vokabular"
assert _test_assignments.loc[30, "Status"] == "Keine positiven Topic-Gewichte"

status_rows = []
for method, assignments in (("LDA", lda_assignments), ("NMF", nmf_assignments)):
    if not assignments.index.equals(df_model.index):
        raise RuntimeError("Die Reihenfolge der Zuordnungen passt nicht zu den Meldungen.")
    if assignments.loc[~model_mask, "Topic"].notna().any():
        raise RuntimeError("Eine Meldung ohne Merkmale hat unzulässigerweise ein Topic erhalten.")
    assigned_count = int(assignments["Topic"].notna().sum())
    unassigned_count = int(assignments["Topic"].isna().sum())
    assert assigned_count + unassigned_count == N_ANALYSIS_DOCS
    status_rows.append({
        "Verfahren": method,
        "Meldungen nach Text Preprocessing": N_ANALYSIS_DOCS,
        "Topic zugeordnet": assigned_count,
        "Nicht zugeordnet": unassigned_count,
        "Davon ohne Merkmale": int((assignments["Status"] == "Keine Merkmale im Vokabular").sum()),
        "Davon ohne positive Topic-Gewichte": int((assignments["Status"] == "Keine positiven Topic-Gewichte").sum()),
        "Anteil nicht zugeordnet in Prozent": round(100 * unassigned_count / N_ANALYSIS_DOCS, 1),
        "Mehrere Topics mit gleichem Höchstwert": int(assignments["Gleicher Höchstwert"].sum()),
    })
    print(f"{method}: {assigned_count} zugeordnet, {unassigned_count} nicht zugeordnet")

assignment_summary = pd.DataFrame(status_rows)
display(assignment_summary)
print("Kontrolle der Zuordnung: bestanden")


In [ ]:
# Schritt 11c Einfluss der Stopwörter vergleichen
# Die Varianten dienen dem Vergleich und ändern die bisherige Analyse nicht.

from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

required_variables = (
    "df_work", "df_model", "df_fit", "df_without_features", "X_count", "X_tfidf",
    "lda_model", "nmf_model", "lda_doc_topic", "nmf_doc_topic",
    "lda_assignments", "nmf_assignments", "count_vectorizer", "tfidf_vectorizer",
    "select_model_documents", "assign_topics", "clean_text",
)
missing_variables = [name for name in required_variables if name not in globals()]
if missing_variables:
    raise RuntimeError(
        "Zuerst die bisherigen Zellen bis einschließlich Schritt 11b ausführen. "
        f"Es fehlen: {', '.join(missing_variables)}"
    )

# Diese im Ausgangszustand entfernten Wörter beschreiben den Zustand oder den Nutzung.
review_status_terms = {
    "defekt", "defekte", "defekten", "defekter",
    "kaputt", "kaputte", "kaputtes", "beschädigt", "beschaedigt",
    "schaden", "schaeden", "schadhaft", "ausgefallen",
    "dunkel", "schwach", "funktion", "funktioniert",
    "brennt", "brennen", "leuchte", "leuchten", "leuchtet",
    "haengt", "hängt", "abgestellt", "abgestellte", "abgestellter",
    "abgestelltes", "abgelegt", "abgeholt", "abholung", "entsorgt",
    "entfernen", "abgelaufen", "abgeschlossen",
}
review_negation_terms = {
    "nicht", "kein", "keine", "keinem", "keinen", "keiner", "keines", "ohne", "nichts",
}
review_original_stopwords = set(STOPWORDS_FINAL)
review_variants = {
    "A Bisherige Stopwörter": review_original_stopwords,
    "B Zustandsangaben behalten": review_original_stopwords - review_status_terms,
    "C Zusätzlich Verneinungen behalten": (
        review_original_stopwords - review_status_terms - review_negation_terms
    ),
}


def clean_text_for_review(text: str, stopwords: set[str]) -> str:
    """Die bisherigen Textregeln mit einer vorgegebenen Menge von Stopwörtern anwenden."""
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-zäöüß\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = []
    for token in text.split():
        token = CANONICAL_TERMS_FINAL.get(token, token)
        if is_location_token(token):
            continue
        if token not in stopwords and len(token) >= MIN_TOKEN_LENGTH:
            tokens.append(token)
    return " ".join(tokens)


# Der Vergleich muss mit denselben Textregeln wie die Hauptanalyse beginnen.
_review_reference = df_work[TEXT_COLUMN].apply(
    lambda text: clean_text_for_review(text, review_original_stopwords)
)
if not _review_reference.equals(df_work[TEXT_COLUMN].apply(clean_text)):
    raise RuntimeError("Die Textregeln des Vergleichs weichen von der Hauptanalyse ab.")

review_rows = []
review_topic_rows = []
review_example_rows = []
review_test_texts = [
    "Lampe defekt", "Ampel funktioniert nicht", "Schild beschädigt",
    "Fahrzeug ohne Kennzeichen", "Fahrrad abgestellt", "Bitte danke",
]

for variant, stopwords in review_variants.items():
    print(f"Prüfe Variante {variant}")
    review_docs = df_work.copy()
    review_docs["text_clean"] = review_docs[TEXT_COLUMN].apply(
        lambda text: clean_text_for_review(text, stopwords)
    )
    review_docs["word_count_clean"] = review_docs["text_clean"].str.split().str.len()
    review_empty = int(review_docs["word_count_clean"].eq(0).sum())
    review_docs = review_docs.loc[
        review_docs["word_count_clean"] >= MIN_TOKENS_CLEAN
    ].reset_index(drop=True)
    if review_docs.empty:
        raise ValueError(f"Variante {variant} enthält keine auswertbaren Texte.")

    cv = CountVectorizer(
        min_df=MIN_DF, max_df=MAX_DF, token_pattern=TOKEN_PATTERN, ngram_range=NGRAM_RANGE
    )
    tv = TfidfVectorizer(
        min_df=MIN_DF, max_df=MAX_DF, token_pattern=TOKEN_PATTERN,
        ngram_range=NGRAM_RANGE, sublinear_tf=SUBLINEAR_TF
    )
    review_count = cv.fit_transform(review_docs["text_clean"])
    review_tfidf = tv.fit_transform(review_docs["text_clean"])
    if not np.array_equal(cv.get_feature_names_out(), tv.get_feature_names_out()):
        raise RuntimeError("Die Vokabulare im Vergleich stimmen nicht überein.")
    review_mask, review_fit, review_count_fit, review_tfidf_fit, review_without = (
        select_model_documents(review_docs, review_count, review_tfidf)
    )
    if N_TOPICS > min(review_tfidf_fit.shape):
        raise ValueError(f"Zu wenige Meldungen oder Merkmale in Variante {variant}.")

    if variant == "A Bisherige Stopwörter":
        if not review_docs[[ID_COLUMN, "text_clean"]].equals(
            df_model[[ID_COLUMN, "text_clean"]]
        ):
            raise RuntimeError("Variante A reproduziert den aktuellen Datenstand nicht.")
        if not np.array_equal(cv.get_feature_names_out(), count_vectorizer.get_feature_names_out()):
            raise RuntimeError("Das Vokabular von Variante A weicht vom aktuellen Stand ab.")
        if (review_count != X_count).nnz or (review_tfidf != X_tfidf).nnz:
            raise RuntimeError("Die Merkmalsmatrizen von Variante A weichen vom aktuellen Stand ab.")
        review_models = {"LDA": lda_model, "NMF": nmf_model}
        review_weights = {"LDA": lda_doc_topic, "NMF": nmf_doc_topic}
    else:
        review_models = {
            "LDA": LatentDirichletAllocation(
                n_components=N_TOPICS, random_state=RANDOM_STATE,
                learning_method="batch", max_iter=LDA_MAX_ITER
            ),
            "NMF": NMF(
                n_components=N_TOPICS, random_state=RANDOM_STATE,
                init="nndsvda", max_iter=NMF_MAX_ITER
            ),
        }
        review_weights = {
            "LDA": review_models["LDA"].fit_transform(review_count_fit),
            "NMF": review_models["NMF"].fit_transform(review_tfidf_fit),
        }

    review_rows.append({
        "Variante": variant,
        "Leere Texte": review_empty,
        "Meldungen nach Text Preprocessing": len(review_docs),
        "Merkmale": review_count.shape[1],
        "Meldungen ohne Merkmale": len(review_without),
        "Meldungen für das Modelltraining": len(review_fit),
    })
    for method, model in review_models.items():
        assignments = assign_topics(review_weights[method], review_mask, review_docs.index)
        if assignments.loc[~review_mask, "Topic"].notna().any():
            raise RuntimeError("Im Vergleich wurde eine Meldung ohne Merkmale einem Topic zugeordnet.")
        feature_names = cv.get_feature_names_out()
        for topic_index, topic_weights in enumerate(model.components_):
            top_indices = topic_weights.argsort()[::-1][:10]
            review_topic_rows.append({
                "Variante": variant,
                "Verfahren": method,
                "Topic": topic_index + 1,
                "Anzahl Meldungen": int(assignments["Topic"].eq(topic_index + 1).sum()),
                "Top-Begriffe": ", ".join(feature_names[top_indices]),
            })
    for text in review_test_texts:
        review_example_rows.append({
            "Variante": variant,
            "Testtext": text,
            "Text nach Preprocessing": clean_text_for_review(text, stopwords),
        })

stopword_review = pd.DataFrame(review_rows)
stopword_review_topics = pd.DataFrame(review_topic_rows)
stopword_review_examples = pd.DataFrame(review_example_rows)

print("\nVergleich der Datengrundlage")
display(stopword_review)
print("\nKünstliche Texte zum Testen, keine Stichprobe aus dem Datensatz")
display(stopword_review_examples)
for variant in review_variants:
    for method in ("LDA", "NMF"):
        print(f"\n{variant} | {method}")
        selected = stopword_review_topics.loc[
            (stopword_review_topics["Variante"] == variant)
            & (stopword_review_topics["Verfahren"] == method),
            ["Topic", "Anzahl Meldungen", "Top-Begriffe"],
        ]
        print(selected.to_string(index=False))

if STOPWORDS_FINAL != review_original_stopwords:
    raise RuntimeError("Die Stopwörter der Hauptanalyse wurden unerwartet verändert.")
print("\nKontrolle des Vergleichs: bestanden")
print("Die Stopwörter und Modelle der Hauptanalyse sind unverändert.")



## Ergebnis des Vergleichs der Stopwörter

Verglichen wurden drei Varianten bei jeweils zehn Topics. Variante A
verwendet die bisherigen Stopwörter. In Variante B bleiben ausgewählte
Zustandsangaben wie „defekt“ und „abgestellt“ erhalten, in Variante C
zusätzlich Verneinungen wie „nicht“ und „ohne“.

Bei NMF erscheinen die Topics von Variante A anhand ihrer wichtigsten
Begriffe am klarsten nach Sachthemen gegliedert. Die bisherigen Stopwörter
werden deshalb vorerst beibehalten. Die Anzahl der Topics und die
Zuordnung einzelner Meldungen werden anschließend geprüft.

In [ ]:
# Schritt 11d verschiedene Anzahl von Topics vergleichen

import warnings
from pathlib import Path

import gensim
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning

# Diese Einstellungen gelten nur für den Vergleich
TOPIC_COUNTS_TO_COMPARE = [6, 8, 10, 12, 15]
COHERENCE_TOP_N = 10
COHERENCE_WINDOW_SIZE = 110


def compare_topic_counts(
    documents, fit_documents, count_matrix, tfidf_matrix, selected_rows,
    count_features, tfidf_features, lda_template, nmf_template,
    topic_counts, top_n, window_size,
):
    """Topic-Modelle auf derselben Datengrundlage trainieren und mit c_v vergleichen."""
    if count_matrix.shape != tfidf_matrix.shape:
        raise ValueError("Count und TF-IDF müssen dieselbe Form haben.")
    if len(fit_documents) != count_matrix.shape[0] or len(fit_documents) == 0:
        raise ValueError("Die Merkmalsmatrizen passen nicht zu den Meldungen für das Training.")
    if not np.array_equal(count_features, tfidf_features):
        raise ValueError("Count und TF-IDF müssen dasselbe Vokabular verwenden.")
    if len(count_features) != count_matrix.shape[1]:
        raise ValueError("Das Vokabular passt nicht zur Zahl der Merkmale.")
    mask = np.asarray(selected_rows)
    if mask.ndim != 1 or mask.dtype != bool or len(mask) != len(documents):
        raise ValueError("Die Auswahl der Meldungen ist ungültig.")
    if not documents.index.is_unique or not documents.index[mask].equals(fit_documents.index):
        raise ValueError("Die Reihenfolge der Meldungen stimmt nicht mit der Auswahl überein.")
    if not documents.loc[mask, "text_clean"].equals(fit_documents["text_clean"]):
        raise ValueError("Die Texte für das Training stimmen nicht mit der Auswahl überein.")
    for matrix in (count_matrix, tfidf_matrix):
        if not np.isfinite(matrix.data).all() or (matrix.data < 0).any():
            raise ValueError("Die Matrizen müssen endliche, nicht negative Werte enthalten.")
        if (np.asarray((matrix != 0).sum(axis=1)).ravel() == 0).any():
            raise ValueError("Die Trainingsmatrizen enthalten Meldungen ohne Merkmale.")
    if not topic_counts or len(set(topic_counts)) != len(topic_counts):
        raise ValueError("Die Anzahl der Topics muss eindeutig vorgegeben sein.")
    if any(isinstance(k, bool) or not isinstance(k, int) or k < 2 for k in topic_counts):
        raise ValueError("Für jeden Versuch sind mindestens zwei Topics erforderlich.")
    if max(topic_counts) > min(tfidf_matrix.shape):
        raise ValueError("Für die NMF-Initialisierung gibt es zu wenige Meldungen oder Merkmale.")
    if isinstance(top_n, bool) or not isinstance(top_n, int) or not 2 <= top_n <= len(count_features):
        raise ValueError("Die Anzahl der Top-Begriffe passt nicht zum Vokabular.")
    if isinstance(window_size, bool) or not isinstance(window_size, int) or window_size < 2:
        raise ValueError("Die Fenstergröße für c_v muss mindestens zwei betragen.")
    if tuple(count_vectorizer.ngram_range) != (1, 1):
        raise ValueError("Dieser Kohärenzvergleich ist für die aktuelle Unigramm-Darstellung ausgelegt.")

    # Alle Varianten verwenden dieselben bereinigten Texte für die Kohärenzberechnung.
    reference_texts = documents["text_clean"].str.split().tolist()
    if any(not isinstance(text, list) or not text for text in reference_texts):
        raise ValueError("Die Referenztexte für c_v dürfen nicht fehlen oder leer sein.")
    dictionary = Dictionary(reference_texts)
    missing_terms = set(count_features) - set(dictionary.token2id)
    if missing_terms:
        raise ValueError("Begriffe aus dem Vokabular fehlen in den Referenztexten für c_v.")

    # Die effektiven LDA-Priors bleiben wie im bisherigen Modell, statt sich mit 1/k zu ändern.
    lda_priors = {
        "doc_topic_prior": float(lda_template.doc_topic_prior_),
        "topic_word_prior": float(lda_template.topic_word_prior_),
    }
    if any(not np.isfinite(value) or value <= 0 for value in lda_priors.values()):
        raise ValueError("Die LDA-Priors müssen positiv und endlich sein.")

    models, weights_by_model, assignments_by_model = {}, {}, {}
    summary_rows, topic_rows = [], []
    print("Meldungen für c_v:", len(documents))
    print("Meldungen je Modelltraining:", len(fit_documents))
    print("Merkmale:", count_matrix.shape[1])
    print("Anzahl der Topics:", topic_counts)
    print("Top-Begriffe je Topic für c_v:", top_n)
    print("Feste LDA-Priors:", lda_priors)

    total_runs = 2 * len(topic_counts)
    completed = 0
    for method, template, matrix in (
        ("LDA", lda_template, count_matrix),
        ("NMF", nmf_template, tfidf_matrix),
    ):
        for k in topic_counts:
            print(f"[{completed + 1}/{total_runs}] {method} mit {k} Topics", flush=True)
            candidate = clone(template).set_params(n_components=k)
            if method == "LDA":
                candidate.set_params(**lda_priors)
            with warnings.catch_warnings(record=True) as recorded_warnings:
                warnings.simplefilter("always", ConvergenceWarning)
                weights = candidate.fit_transform(matrix)
            warning_messages = [str(warning.message) for warning in recorded_warnings]
            for message in warning_messages:
                print(f"  Warnung: {message}")

            components = candidate.components_
            if components.shape != (k, len(count_features)):
                raise RuntimeError("Die Modell-Komponenten passen nicht zum Vergleich.")
            if not np.isfinite(components).all() or (components < 0).any():
                raise RuntimeError("Das Modell enthält ungültige Gewichte.")
            if ((components > 0).sum(axis=1) < top_n).any():
                raise RuntimeError("Ein Topic enthält zu wenige positive Gewichte für die Top-Begriffe.")
            top_indices = np.argsort(-components, axis=1, kind="stable")[:, :top_n]
            topics = [count_features[indices].tolist() for indices in top_indices]

            # Ein Prozess vermeidet zusätzliche Worker bei der Ausführung unter Windows.
            coherence = CoherenceModel(
                topics=topics, texts=reference_texts, dictionary=dictionary,
                coherence="c_v", topn=top_n, window_size=window_size, processes=1,
            )
            per_topic = np.asarray(coherence.get_coherence_per_topic(), dtype=float)
            if per_topic.shape != (k,) or not np.isfinite(per_topic).all():
                raise RuntimeError(f"Für {method} mit {k} Topics ist c_v nicht vollständig berechenbar.")

            assignments = assign_topics(weights, mask, documents.index)
            if assignments.loc[~mask, "Topic"].notna().any():
                raise RuntimeError("Eine Meldung ohne Merkmale wurde einem Topic zugeordnet.")
            counts = assignments["Topic"].value_counts().reindex(range(1, k + 1), fill_value=0)
            assigned = int(assignments["Topic"].notna().sum())
            unassigned = int(assignments["Topic"].isna().sum())
            if int(counts.sum()) != assigned or assigned + unassigned != len(documents):
                raise RuntimeError("Die Häufigkeiten des Vergleichs sind nicht konsistent.")

            key = (method, k)
            models[key] = candidate
            weights_by_model[key] = weights
            assignments_by_model[key] = assignments
            summary_rows.append({
                "Verfahren": method,
                "Anzahl Topics": k,
                "c_v Mittelwert": float(per_topic.mean()),
                "c_v Minimum": float(per_topic.min()),
                "Zugeordnet": assigned,
                "Nicht zugeordnet": unassigned,
                "Topics ohne Zuordnung": int(counts.eq(0).sum()),
                "Iterationen": int(candidate.n_iter_),
                "Warnungen": len(warning_messages),
                "Hinweise": " | ".join(warning_messages),
            })
            for position, (terms, value) in enumerate(zip(topics, per_topic), start=1):
                topic_rows.append({
                    "Verfahren": method,
                    "Anzahl Topics": k,
                    "Topic": position,
                    "Anzahl Meldungen": int(counts.loc[position]),
                    "Anteil Prozent": 100 * int(counts.loc[position]) / len(documents),
                    "Bezugsmenge Meldungen": len(documents),
                    "c_v": float(value),
                    "Top-Begriffe": ", ".join(terms),
                })
            completed += 1
            print(f"  c_v Mittelwert: {per_topic.mean():.4f}, nicht zugeordnet: {unassigned}", flush=True)

    return (
        pd.DataFrame(summary_rows), pd.DataFrame(topic_rows),
        models, weights_by_model, assignments_by_model,
    )


required_for_topic_comparison = (
    "df_model", "df_fit", "model_mask", "X_count_fit", "X_tfidf_fit",
    "count_vectorizer", "tfidf_vectorizer", "lda_model", "nmf_model",
    "assign_topics", "OUTPUT_DIR",
)
missing_for_topic_comparison = [
    name for name in required_for_topic_comparison if name not in globals()
]
if missing_for_topic_comparison:
    raise RuntimeError(
        "Zuerst alle Zellen bis einschließlich Schritt 11b ausführen. "
        f"Es fehlen: {', '.join(missing_for_topic_comparison)}"
    )

(
    topic_count_review, topic_count_details,
    topic_count_models, topic_count_weights, topic_count_assignments,
) = compare_topic_counts(
    df_model, df_fit, X_count_fit, X_tfidf_fit, model_mask,
    count_vectorizer.get_feature_names_out(), tfidf_vectorizer.get_feature_names_out(),
    lda_model, nmf_model, TOPIC_COUNTS_TO_COMPARE, COHERENCE_TOP_N, COHERENCE_WINDOW_SIZE,
)

print("\nVergleich der Anzahl von Topics")
display(topic_count_review.drop(columns=["Hinweise"]).round(4))

# Nur Vergleichswerte und Top-Begriffe exportieren, keine originalen Meldungen.
comparison_dir = Path(OUTPUT_DIR)
comparison_dir.mkdir(parents=True, exist_ok=True)
comparison_path = comparison_dir / "vergleich_topic_anzahl.csv"
comparison_topics_path = comparison_dir / "vergleich_topic_anzahl_details.csv"
topic_count_review.to_csv(comparison_path, index=False, sep=";", encoding="utf-8-sig")
topic_count_details.to_csv(comparison_topics_path, index=False, sep=";", encoding="utf-8-sig")

print("Gespeichert:")
print(comparison_path.resolve())
print(comparison_topics_path.resolve())
print("Kontrolle des Vergleichs: bestanden")
print("Die Modelle und Zuordnungen der Hauptanalyse sind unverändert.")


In [ ]:
# Schritt 11e Meldungen ausgewählter Modelle zur inhaltlichen Prüfung zusammenstellen

from pathlib import Path

import numpy as np
import pandas as pd

# Die Kandidaten werden nur geprüft, die Hauptanalyse bleibt unverändert.
MODELS_TO_REVIEW = [("NMF", 12), ("NMF", 15), ("LDA", 10), ("LDA", 15)]
EXAMPLES_BY_WEIGHT = 2
RANDOM_EXAMPLES = 3


def collect_topic_examples(
    documents, fit_documents, selected_rows, models, weights_by_model,
    assignments_by_model, details, candidates, text_column, id_column,
    n_by_weight=2, n_random=3, random_state=42,
):
    """Pro Topic die stark gewichteten Meldungen und eine getrennte Zufallsauswahl zusammenstellen."""
    required_columns = {text_column, id_column, "category_parent", "category_child"}
    if not required_columns.issubset(documents.columns):
        raise ValueError("Textspalte, ID oder Kategorien fehlen in den Meldungen.")
    if not documents.index.is_unique or not fit_documents.index.is_unique:
        raise ValueError("Die Indizes der Meldungen müssen eindeutig sein.")
    if documents[id_column].isna().any() or documents[id_column].duplicated().any():
        raise ValueError("Die Melde-IDs müssen vorhanden und eindeutig sein.")
    if any(isinstance(n, bool) or not isinstance(n, int) or n < 0 for n in (n_by_weight, n_random)):
        raise ValueError("Die Anzahl der Beispiele muss jeweils eine nicht negative ganze Zahl sein.")
    if n_by_weight + n_random == 0:
        raise ValueError("Mindestens eine Meldung pro Topic muss ausgewählt werden.")
    if not candidates or len(set(candidates)) != len(candidates):
        raise ValueError("Die Liste der zu prüfenden Modelle darf nicht leer sein oder Einträge wiederholen.")

    mask = np.asarray(selected_rows)
    if mask.ndim != 1 or mask.dtype != bool or len(mask) != len(documents):
        raise ValueError("Die Auswahl der Meldungen passt nicht zum Datensatz.")
    if not documents.index[mask].equals(fit_documents.index):
        raise ValueError("Die Reihenfolge der Meldungen stimmt nicht mit den Modellgewichten überein.")
    if not documents.loc[mask, "text_clean"].equals(fit_documents["text_clean"]):
        raise ValueError("Die Texte wurden seit dem Modellvergleich verändert.")

    rng = np.random.default_rng(random_state)
    rows = []
    overview = []
    fit_index = fit_documents.index

    for method, k in candidates:
        key = (method, k)
        if any(key not in collection for collection in (models, weights_by_model, assignments_by_model)):
            raise ValueError(f"Das Vergleichsmodell {method} mit {k} Topics fehlt. Zuerst Schritt 11d ausführen.")
        model = models[key]
        weights = np.asarray(weights_by_model[key], dtype=float)
        assignments = assignments_by_model[key]
        if model.n_components != k or weights.shape != (len(fit_documents), k):
            raise ValueError(f"Die Modellgewichte von {method} mit {k} Topics haben eine unerwartete Form.")
        if not np.isfinite(weights).all() or (weights < 0).any():
            raise ValueError("Die Modellgewichte müssen endlich und nicht negativ sein.")
        if not assignments.index.equals(documents.index):
            raise ValueError("Die Zuordnungen passen nicht zum Index der Meldungen.")
        if assignments.loc[~mask, "Topic"].notna().any():
            raise ValueError("Eine Meldung ohne Merkmale hat eine Topic-Zuordnung.")

        # Die gespeicherten Zuordnungen müssen zu den vorhandenen Gewichtungen passen.
        expected = pd.Series(pd.NA, index=fit_index, dtype="Int64")
        positive = weights.max(axis=1) > 0
        expected.loc[fit_index[positive]] = weights[positive].argmax(axis=1) + 1
        actual = assignments.loc[fit_index, "Topic"].astype("Int64")
        if not np.array_equal(expected.to_numpy(na_value=-1), actual.to_numpy(na_value=-1)):
            raise ValueError("Modellgewichte und Topic-Zuordnungen stimmen nicht überein.")

        model_details = details.loc[
            (details["Verfahren"] == method) & (details["Anzahl Topics"] == k)
        ].set_index("Topic")
        if not model_details.index.is_unique or set(model_details.index) != set(range(1, k + 1)):
            raise ValueError("Die Detailtabelle enthält nicht genau einen Eintrag pro Topic.")

        before_model = len(rows)
        for topic in range(1, k + 1):
            positions = np.flatnonzero(actual.eq(topic).fillna(False).to_numpy(dtype=bool))
            detail = model_details.loc[topic]
            if len(positions) != int(detail["Anzahl Meldungen"]):
                raise ValueError("Die Häufigkeiten der Detailtabelle passen nicht zu den Zuordnungen.")

            # Hohe Gewichtungen dienen nur zur Auswahl von Beispielen, nicht als Beweis einer korrekten Zuordnung.
            order = np.argsort(-weights[positions, topic - 1], kind="stable")
            strongest = positions[order[:min(n_by_weight, len(positions))]]
            remaining = positions[~np.isin(positions, strongest)]
            random_selection = rng.choice(remaining, size=min(n_random, len(remaining)), replace=False)
            selections = [(int(p), "Hohes Topic-Gewicht") for p in strongest]
            selections += [(int(p), "Zufallsauswahl") for p in random_selection]
            if len({p for p, _ in selections}) != len(selections):
                raise RuntimeError("Eine Meldung wurde innerhalb eines Topics mehrfach ausgewählt.")

            for position, selection in selections:
                record = documents.loc[fit_index[position]]
                rows.append({
                    "Verfahren": method,
                    "Anzahl Topics": k,
                    "Topic": topic,
                    "Top-Begriffe": detail["Top-Begriffe"],
                    "c_v": float(detail["c_v"]),
                    "Meldungen im Topic": len(positions),
                    "Auswahl": selection,
                    "ID": record[id_column],
                    "Meldetext": record[text_column],
                    "Hauptkategorie": record["category_parent"],
                    "Unterkategorie": record["category_child"],
                })
        overview.append({"Verfahren": method, "Anzahl Topics": k, "Ausgewählte Meldungen": len(rows) - before_model})

    columns = [
        "Verfahren", "Anzahl Topics", "Topic", "Top-Begriffe", "c_v",
        "Meldungen im Topic", "Auswahl", "ID", "Meldetext", "Hauptkategorie", "Unterkategorie",
    ]
    return pd.DataFrame(rows, columns=columns), pd.DataFrame(overview)


required_for_examples = (
    "df_model", "df_fit", "model_mask", "topic_count_models", "topic_count_weights",
    "topic_count_assignments", "topic_count_details", "TEXT_COLUMN", "ID_COLUMN", "OUTPUT_DIR",
)
missing_for_examples = [name for name in required_for_examples if name not in globals()]
if missing_for_examples:
    raise RuntimeError(
        "Zuerst die Hauptanalyse bis Schritt 11b und den Modellvergleich in Schritt 11d ausführen. "
        f"Es fehlen: {', '.join(missing_for_examples)}"
    )

review_topic_examples, review_example_counts = collect_topic_examples(
    df_model, df_fit, model_mask,
    topic_count_models, topic_count_weights, topic_count_assignments, topic_count_details,
    MODELS_TO_REVIEW, TEXT_COLUMN, ID_COLUMN,
    n_by_weight=EXAMPLES_BY_WEIGHT, n_random=RANDOM_EXAMPLES, random_state=RANDOM_STATE,
)

# Die Datei enthält die Originalmeldungen und bleibt im lokalen Ausgabeordner.
example_path = Path(OUTPUT_DIR) / "pruefung_topic_meldungen.csv"
example_path.parent.mkdir(parents=True, exist_ok=True)
review_topic_examples.to_csv(example_path, index=False, sep=";", encoding="utf-8-sig")
print(review_example_counts.to_string(index=False))
print("Insgesamt ausgewählte Meldungen:", len(review_topic_examples))
print("Kontrolle der Auswahl: bestanden")
print("Gespeichert:", example_path.resolve())


In [ ]:
# Schritt 11f NMF-Zuordnung bei einheitlicher Skalierung vergleichen
# Die vorhandenen Modelle und ihre Zuordnungen bleiben unverändert.

from pathlib import Path
import numpy as np
import pandas as pd


def normalize_nmf_for_assignment(coefficients, components):
    """Topic-Vektoren auf L2-Norm 1 bringen und die Koeffizienten entsprechend anpassen."""
    w = np.asarray(coefficients, dtype=float)
    h = np.asarray(components, dtype=float)
    if w.ndim != 2 or h.ndim != 2 or w.shape[1] != h.shape[0] or h.shape[0] == 0:
        raise ValueError("Die beiden NMF-Matrizen passen nicht zusammen.")
    if h.shape[1] == 0:
        raise ValueError("Die Topic-Vektoren enthalten keine Merkmale.")
    if any(not np.isfinite(a).all() or (a < 0).any() for a in (w, h)):
        raise ValueError("NMF-Koeffizienten müssen endlich und nicht negativ sein.")

    lengths = np.linalg.norm(h, axis=1)
    if not np.isfinite(lengths).all() or (lengths <= 0).any():
        raise ValueError("Mindestens ein Topic-Vektor hat keine verwendbare L2-Norm.")
    h_unit = h / lengths[:, None]
    w_scaled = w * lengths[None, :]
    if not np.isfinite(w_scaled).all():
        raise ValueError("Die angepassten NMF-Koeffizienten enthalten ungültige Werte.")
    if not np.allclose(np.linalg.norm(h_unit, axis=1), 1.0):
        raise RuntimeError("Die Topic-Vektoren haben nicht die erwartete L2-Norm.")

    # Das Produkt beider Matrizen muss erhalten bleiben, kleine Abweichungen in der Rundung sind zulässig.
    for start in range(0, len(w), 256):
        before = w[start:start + 256] @ h
        after = w_scaled[start:start + 256] @ h_unit
        if not np.allclose(before, after, rtol=1e-10, atol=1e-12):
            raise RuntimeError("Die Rekonstruktion wurde durch die Skalierung verändert.")
    return w_scaled, h_unit


# Gleiche Rekonstruktion bei unterschiedlicher Skalierung muss gleiche angepasste Werte ergeben.
_scale_test_w = np.array([[1.0, 2.0], [0.0, 0.0]])
_scale_test_h = np.array([[10.0, 0.0], [0.0, 1.0]])
_scale_test_a, _ = normalize_nmf_for_assignment(_scale_test_w, _scale_test_h)
_scale_test_factors = np.array([0.01, 100.0])
_scale_test_b, _ = normalize_nmf_for_assignment(
    _scale_test_w / _scale_test_factors,
    _scale_test_h * _scale_test_factors[:, None],
)
if not np.allclose(_scale_test_a, _scale_test_b) or np.any(_scale_test_a[1] != 0):
    raise RuntimeError("Der Funktionstest der NMF-Skalierung ist fehlgeschlagen.")

_scale_required = (
    "df_model", "df_fit", "model_mask", "topic_count_models", "topic_count_weights",
    "topic_count_assignments", "topic_count_details", "review_topic_examples",
    "assign_topics", "TEXT_COLUMN", "ID_COLUMN", "OUTPUT_DIR", "RANDOM_STATE",
)
_scale_missing = [name for name in _scale_required if name not in globals()]
if _scale_missing:
    raise RuntimeError("Zuerst die Zellen bis Schritt 11e ausführen. Es fehlen: " + ", ".join(_scale_missing))
if not df_model.index[model_mask].equals(df_fit.index):
    raise ValueError("Die Zeilenauswahl stimmt nicht mit den Meldungen für das Training überein.")
if not df_model.loc[model_mask, "text_clean"].equals(df_fit["text_clean"]):
    raise ValueError("Die Texte der Hauptanalyse wurden verändert.")
if df_model[ID_COLUMN].isna().any() or df_model[ID_COLUMN].duplicated().any():
    raise ValueError("Die Melde-IDs müssen vorhanden und eindeutig sein.")

nmf_scale_rows = []
nmf_scale_example_rows = []
nmf_scale_assignments = {}
nmf_scale_coefficients = {}
nmf_scale_components = {}

for _scale_k in (12, 15):
    _scale_key = ("NMF", _scale_k)
    if any(_scale_key not in collection for collection in (
        topic_count_models, topic_count_weights, topic_count_assignments
    )):
        raise ValueError(f"Das NMF-Vergleichsmodell mit {_scale_k} Topics fehlt.")
    _scale_model = topic_count_models[_scale_key]
    _scale_w = np.asarray(topic_count_weights[_scale_key], dtype=float)
    _scale_h = np.asarray(_scale_model.components_, dtype=float)
    if _scale_w.shape != (len(df_fit), _scale_k):
        raise ValueError("Die NMF-Koeffizienten passen nicht zur Zahl der Meldungen und Topics.")
    _scale_w_before, _scale_h_before = _scale_w.copy(), _scale_h.copy()
    _scale_raw = assign_topics(_scale_w, model_mask, df_model.index)
    if not _scale_raw.equals(topic_count_assignments[_scale_key]):
        raise ValueError("Die gespeicherten NMF-Zuordnungen passen nicht zu den Modellwerten.")

    _scale_w_new, _scale_h_new = normalize_nmf_for_assignment(_scale_w, _scale_h)
    _scale_new = assign_topics(_scale_w_new, model_mask, df_model.index)
    if not _scale_new["Topic"].isna().equals(_scale_raw["Topic"].isna()):
        raise RuntimeError("Der Status nicht zugeordneter Meldungen wurde verändert.")
    if _scale_new.loc[~model_mask, "Topic"].notna().any():
        raise RuntimeError("Eine Meldung ohne Merkmale hat eine Topic-Zuordnung erhalten.")
    if not np.array_equal(_scale_w, _scale_w_before) or not np.array_equal(_scale_h, _scale_h_before):
        raise RuntimeError("Die ursprünglichen NMF-Matrizen wurden verändert.")

    _scale_changed = _scale_raw["Topic"].fillna(-1).ne(_scale_new["Topic"].fillna(-1))
    nmf_scale_assignments[_scale_key] = _scale_new
    nmf_scale_coefficients[_scale_key] = _scale_w_new
    nmf_scale_components[_scale_key] = _scale_h_new
    nmf_scale_rows.append({
        "Anzahl Topics": _scale_k,
        "Zugeordnet": int(_scale_new["Topic"].notna().sum()),
        "Nicht zugeordnet": int(_scale_new["Topic"].isna().sum()),
        "Andere Topic-Zuordnung": int(_scale_changed.sum()),
        "Rekonstruktion unverändert": True,
    })

    _scale_details = topic_count_details.loc[
        (topic_count_details["Verfahren"] == "NMF")
        & (topic_count_details["Anzahl Topics"] == _scale_k)
    ].set_index("Topic")
    if not _scale_details.index.is_unique or set(_scale_details.index) != set(range(1, _scale_k + 1)):
        raise ValueError("Die Detailtabelle enthält nicht genau einen Eintrag pro Topic.")

    # Die bereits ausgewählten originalen Meldungen bleiben für den Vergleich dieselben.
    _scale_sample = review_topic_examples.loc[
        (review_topic_examples["Verfahren"] == "NMF")
        & (review_topic_examples["Anzahl Topics"] == _scale_k)
    ].copy()
    if _scale_sample.empty or _scale_sample["ID"].duplicated().any():
        raise ValueError("Die bisherige Stichprobe fehlt oder enthält doppelte Melde-IDs.")
    _scale_id_to_index = pd.Series(df_model.index, index=df_model[ID_COLUMN])
    if not _scale_sample["ID"].isin(_scale_id_to_index.index).all():
        raise ValueError("Eine Melde-ID aus der Stichprobe fehlt in der Hauptanalyse.")
    _scale_selected = {}
    for _, _scale_record in _scale_sample.iterrows():
        _scale_index = _scale_id_to_index.loc[_scale_record["ID"]]
        if df_model.loc[_scale_index, TEXT_COLUMN] != _scale_record["Meldetext"]:
            raise ValueError("Der Originaltext einer ausgewählten Meldung wurde verändert.")
        if int(_scale_raw.loc[_scale_index, "Topic"]) != int(_scale_record["Topic"]):
            raise ValueError("Die Stichprobe gehört zu einer anderen Topic-Zuordnung.")
        _scale_selected[_scale_index] = "Bisherige Stichprobe"

    # Zusätzlich bis zu zehn geänderte Zuordnungen außerhalb der bisherigen Stichprobe auswählen.
    _scale_remaining = df_model.index[_scale_changed].difference(pd.Index(_scale_selected))
    _scale_rng = np.random.default_rng(RANDOM_STATE + _scale_k)
    for _scale_index in _scale_rng.choice(
        _scale_remaining.to_numpy(), size=min(10, len(_scale_remaining)), replace=False
    ):
        _scale_selected[_scale_index] = "Zufallsauswahl geänderter Zuordnungen"

    for _scale_index, _scale_selection in _scale_selected.items():
        _scale_record = df_model.loc[_scale_index]
        _scale_old_topic = int(_scale_raw.loc[_scale_index, "Topic"])
        _scale_new_topic = int(_scale_new.loc[_scale_index, "Topic"])
        nmf_scale_example_rows.append({
            "Anzahl Topics": _scale_k,
            "ID": _scale_record[ID_COLUMN],
            "Auswahl": _scale_selection,
            "Meldetext": _scale_record[TEXT_COLUMN],
            "Text nach Preprocessing": _scale_record["text_clean"],
            "Topic bisher": _scale_old_topic,
            "Topic nach Skalierung": _scale_new_topic,
            "Top-Begriffe bisher": _scale_details.loc[_scale_old_topic, "Top-Begriffe"],
            "Top-Begriffe nach Skalierung": _scale_details.loc[_scale_new_topic, "Top-Begriffe"],
            "Zuordnung geändert": _scale_old_topic != _scale_new_topic,
        })

nmf_scale_summary = pd.DataFrame(nmf_scale_rows)
nmf_scale_examples = pd.DataFrame(nmf_scale_example_rows)
_scale_output_dir = Path(OUTPUT_DIR)
_scale_output_dir.mkdir(parents=True, exist_ok=True)
_scale_summary_path = _scale_output_dir / "vergleich_nmf_skalierung.csv"
_scale_examples_path = _scale_output_dir / "pruefung_nmf_skalierung_meldungen.csv"
nmf_scale_summary.to_csv(_scale_summary_path, index=False, sep=";", encoding="utf-8-sig")
# Diese Datei enthält Originalmeldungen und bleibt lokal.
nmf_scale_examples.to_csv(_scale_examples_path, index=False, sep=";", encoding="utf-8-sig")
print(nmf_scale_summary.to_string(index=False))
print("Technische Kontrolle der NMF-Skalierung: bestanden")
print("Die Modelle und Zuordnungen der Hauptanalyse sind unverändert.")
print("Gespeichert:", _scale_summary_path.resolve())
print("Gespeichert:", _scale_examples_path.resolve())


In [ ]:
# Schritt 11g Die Ergänzung von "leuchte" zu den Begriffszuordnungen prüfen
# Die Konfiguration, Modelle und Zuordnungen der Hauptanalyse bleiben unverändert.

from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning

_leuchte_required = (
    "df_work", "df_model", "df_fit", "df_without_features", "X_count",
    "count_vectorizer", "tfidf_vectorizer", "topic_count_models",
    "nmf_scale_assignments", "normalize_nmf_for_assignment", "assign_topics",
    "select_model_documents", "clean_text", "is_location_token", "STOPWORDS_FINAL",
    "CANONICAL_TERMS_FINAL", "MIN_TOKEN_LENGTH", "MIN_TOKENS_CLEAN",
    "TEXT_COLUMN", "ID_COLUMN", "OUTPUT_DIR",
)
_leuchte_missing = [name for name in _leuchte_required if name not in globals()]
if _leuchte_missing:
    raise RuntimeError("Zuerst die Zellen bis Schritt 11f ausführen. Es fehlen: " + ", ".join(_leuchte_missing))
if CANONICAL_TERMS_FINAL.get("leuchte") == "beleuchtung":
    raise ValueError('"leuchte" ist bereits zugeordnet. Dieser Vergleich benötigt die bisherige Konfiguration.')
if not df_work[ID_COLUMN].is_unique or not df_model[ID_COLUMN].is_unique:
    raise ValueError("Für den Vergleich müssen die Melde-IDs eindeutig sein.")


def preprocess_with_terms(text: str, terms: dict[str, str]) -> str:
    """Die vorhandenen Textregeln mit einer vorgegebenen Begriffszuordnung anwenden."""
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-zäöüß\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = []
    for token in text.split():
        token = terms.get(token, token)
        if is_location_token(token):
            continue
        if token not in STOPWORDS_FINAL and len(token) >= MIN_TOKEN_LENGTH:
            tokens.append(token)
    return " ".join(tokens)


_leuchte_terms_before = dict(CANONICAL_TERMS_FINAL)
_leuchte_stopwords_before = set(STOPWORDS_FINAL)
_leuchte_before = df_work[TEXT_COLUMN].apply(clean_text)
_leuchte_reference = df_work[TEXT_COLUMN].apply(
    lambda text: preprocess_with_terms(text, _leuchte_terms_before)
)
if not _leuchte_reference.equals(_leuchte_before):
    raise RuntimeError("Die Textregeln des Vergleichs stimmen nicht mit der Hauptanalyse überein.")

# Nur diese eine Ergänzung untersuchen. Die Form "leuchten" bleibt unverändert.
_leuchte_terms = {**_leuchte_terms_before, "leuchte": "beleuchtung"}
_leuchte_after = df_work[TEXT_COLUMN].apply(
    lambda text: preprocess_with_terms(text, _leuchte_terms)
)
_leuchte_changed = _leuchte_before.ne(_leuchte_after)
if not _leuchte_changed.any():
    raise ValueError("Die Ergänzung verändert bei diesen Daten keine Meldung.")
if not _leuchte_after.loc[_leuchte_changed].str.split().apply(lambda tokens: "beleuchtung" in tokens).all():
    raise RuntimeError('Der Begriff "beleuchtung" fehlt nach der Ergänzung.')

_leuchte_docs = df_work.copy()
_leuchte_docs["text_clean"] = _leuchte_after
_leuchte_docs["word_count_clean"] = _leuchte_after.str.split().str.len()
_leuchte_docs = _leuchte_docs.loc[
    _leuchte_docs["word_count_clean"] >= MIN_TOKENS_CLEAN
].reset_index(drop=True)
if _leuchte_docs.empty:
    raise ValueError("Nach dem Text Preprocessing sind keine Meldungen vorhanden.")

_leuchte_cv = clone(count_vectorizer)
_leuchte_tv = clone(tfidf_vectorizer)
_leuchte_count = _leuchte_cv.fit_transform(_leuchte_docs["text_clean"])
_leuchte_tfidf = _leuchte_tv.fit_transform(_leuchte_docs["text_clean"])
_leuchte_features = _leuchte_cv.get_feature_names_out()
if not np.array_equal(_leuchte_features, _leuchte_tv.get_feature_names_out()):
    raise RuntimeError("Count und TF-IDF verwenden unterschiedliche Vokabulare.")
(
    _leuchte_mask, _leuchte_fit, _leuchte_count_fit, _leuchte_tfidf_fit, _leuchte_without,
) = select_model_documents(_leuchte_docs, _leuchte_count, _leuchte_tfidf)

leuchte_summary = pd.DataFrame([
    {"Variante": "Bisher", "Meldungen nach Text Preprocessing": len(df_model),
     "Merkmale": X_count.shape[1], "Ohne Merkmale": len(df_without_features),
     "Für das Modelltraining": len(df_fit)},
    {"Variante": "Mit leuchte -> beleuchtung", "Meldungen nach Text Preprocessing": len(_leuchte_docs),
     "Merkmale": _leuchte_count.shape[1], "Ohne Merkmale": len(_leuchte_without),
     "Für das Modelltraining": len(_leuchte_fit)},
])
print("Meldungen mit verändertem Text:", int(_leuchte_changed.sum()))
print(leuchte_summary.to_string(index=False))

_leuchte_old_index = pd.Series(df_model.index, index=df_model[ID_COLUMN])
_leuchte_new_index = pd.Series(_leuchte_docs.index, index=_leuchte_docs[ID_COLUMN])
_leuchte_rows = []
leuchte_models, leuchte_assignments = {}, {}
for _leuchte_k in (12, 15):
    _leuchte_key = ("NMF", _leuchte_k)
    if _leuchte_key not in topic_count_models or _leuchte_key not in nmf_scale_assignments:
        raise ValueError(f"Das NMF-Vergleichsmodell mit {_leuchte_k} Topics fehlt.")
    print(f"NMF mit {_leuchte_k} Topics und ergänzter Begriffszuordnung", flush=True)
    _leuchte_model = clone(topic_count_models[_leuchte_key])
    with warnings.catch_warnings(record=True) as _leuchte_warnings:
        warnings.simplefilter("always", ConvergenceWarning)
        _leuchte_w = _leuchte_model.fit_transform(_leuchte_tfidf_fit)
    for _leuchte_warning in _leuchte_warnings:
        print("Warnung:", str(_leuchte_warning.message))

    # Auf beiden Seiten dieselbe L2-Skalierung aus Schritt 11f verwenden.
    _leuchte_w_scaled, _ = normalize_nmf_for_assignment(_leuchte_w, _leuchte_model.components_)
    _leuchte_assignments = assign_topics(_leuchte_w_scaled, _leuchte_mask, _leuchte_docs.index)
    if _leuchte_assignments.loc[~_leuchte_mask, "Topic"].notna().any():
        raise RuntimeError("Eine Meldung ohne Merkmale wurde zugeordnet.")
    leuchte_models[_leuchte_key] = _leuchte_model
    leuchte_assignments[_leuchte_key] = _leuchte_assignments

    _leuchte_old_h = topic_count_models[_leuchte_key].components_
    _leuchte_old_features = tfidf_vectorizer.get_feature_names_out()
    for _leuchte_position in df_work.index[_leuchte_changed]:
        _leuchte_record = df_work.loc[_leuchte_position]
        _leuchte_id = _leuchte_record[ID_COLUMN]
        _leuchte_old_topic, _leuchte_old_terms = pd.NA, ""
        _leuchte_old_status = "Durch Textfilter ausgeschlossen"
        if _leuchte_id in _leuchte_old_index.index:
            _leuchte_old_assignment = nmf_scale_assignments[_leuchte_key].loc[_leuchte_old_index.loc[_leuchte_id]]
            _leuchte_old_topic = _leuchte_old_assignment["Topic"]
            _leuchte_old_status = _leuchte_old_assignment["Status"]
            if pd.notna(_leuchte_old_topic):
                _leuchte_indices = np.argsort(-_leuchte_old_h[int(_leuchte_old_topic) - 1], kind="stable")[:10]
                _leuchte_old_terms = ", ".join(_leuchte_old_features[_leuchte_indices])

        _leuchte_new_assignment = _leuchte_assignments.loc[_leuchte_new_index.loc[_leuchte_id]]
        _leuchte_new_topic = _leuchte_new_assignment["Topic"]
        _leuchte_new_terms = ""
        if pd.notna(_leuchte_new_topic):
            _leuchte_indices = np.argsort(-_leuchte_model.components_[int(_leuchte_new_topic) - 1], kind="stable")[:10]
            _leuchte_new_terms = ", ".join(_leuchte_features[_leuchte_indices])
        _leuchte_rows.append({
            "Anzahl Topics": _leuchte_k, "ID": _leuchte_id,
            "Meldetext": _leuchte_record[TEXT_COLUMN],
            "Text bisher": _leuchte_before.loc[_leuchte_position],
            "Text nach Ergänzung": _leuchte_after.loc[_leuchte_position],
            "Status bisher": _leuchte_old_status,
            "Status nach Ergänzung": _leuchte_new_assignment["Status"],
            "Topic bisher": _leuchte_old_topic, "Topic nach Ergänzung": _leuchte_new_topic,
            "Top-Begriffe bisher": _leuchte_old_terms,
            "Top-Begriffe nach Ergänzung": _leuchte_new_terms,
        })

# Topic-Nummern verschiedener Trainingsläufe sind keine festen Themenbezeichnungen.
leuchte_examples = pd.DataFrame(_leuchte_rows)
for _leuchte_column in ("Topic bisher", "Topic nach Ergänzung"):
    leuchte_examples[_leuchte_column] = leuchte_examples[_leuchte_column].astype("Int64")
if leuchte_examples.duplicated(["Anzahl Topics", "ID"]).any():
    raise RuntimeError("Eine Meldung kommt pro Modell mehrfach in der Prüftabelle vor.")
if len(leuchte_examples) != 2 * int(_leuchte_changed.sum()):
    raise RuntimeError("Nicht alle betroffenen Meldungen wurden für beide Modelle erfasst.")
if CANONICAL_TERMS_FINAL != _leuchte_terms_before or STOPWORDS_FINAL != _leuchte_stopwords_before:
    raise RuntimeError("Die Konfiguration der Hauptanalyse wurde verändert.")

_leuchte_output_dir = Path(OUTPUT_DIR)
_leuchte_output_dir.mkdir(parents=True, exist_ok=True)
leuchte_summary.to_csv(_leuchte_output_dir / "vergleich_leuchte.csv", index=False, sep=";", encoding="utf-8-sig")
# Die zweite Datei enthält die originalen Meldungen und bleibt lokal.
leuchte_examples.to_csv(_leuchte_output_dir / "pruefung_leuchte_meldungen.csv", index=False, sep=";", encoding="utf-8-sig")
print("Ausgewählte Meldungen für den Vergleich:", len(leuchte_examples))
print("Technische Kontrolle der Begriffszuordnung: bestanden")
print("Die Konfiguration, Modelle und Zuordnungen der Hauptanalyse sind unverändert.")
print("Gespeichert:", (_leuchte_output_dir / "vergleich_leuchte.csv").resolve())
print("Gespeichert:", (_leuchte_output_dir / "pruefung_leuchte_meldungen.csv").resolve())


In [ ]:
# Schritt 12 Wichtigste Begriffe je Topic anzeigen

def print_top_words(model, feature_names, n_top_words=12, title=""):
    print("\n" + title)
    print("=" * len(title))

    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[::-1][:n_top_words]
        top_words = [feature_names[i] for i in top_indices]
        print(f"Topic {topic_idx + 1}: {', '.join(top_words)}")

print_top_words(
    lda_model,
    count_vectorizer.get_feature_names_out(),
    n_top_words=12,
    title="LDA Topics auf CountVectorizer"
)

print_top_words(
    nmf_model,
    tfidf_vectorizer.get_feature_names_out(),
    n_top_words=12,
    title="NMF Topics auf TfidfVectorizer"
)


In [ ]:
# Schritt 13 Technische Themenübersichten erstellen

def build_topic_summary(model, feature_names, assignments, source_df, n_top_words=12, n_examples=2):
    if not assignments.index.equals(source_df.index):
        raise ValueError("Meldungen und Topic-Zuordnungen haben nicht denselben Index.")
    n_docs = len(source_df)
    if n_docs == 0:
        raise ValueError("Die Bezugsmenge für die Prozentanteile ist leer.")

    rows = []

    for topic_idx in range(model.n_components):
        topic_number = topic_idx + 1

        # Top Begriffe
        topic_weights = model.components_[topic_idx]
        top_indices = topic_weights.argsort()[::-1][:n_top_words]
        top_words = ", ".join(feature_names[top_indices])

        # Dokumente mit dominantem Topic
        doc_indices = np.flatnonzero(
            assignments["Topic"].eq(topic_number).fillna(False).to_numpy(dtype=bool)
        )
        doc_count = len(doc_indices)
        doc_share = round((doc_count / n_docs) * 100, 1)

        # Häufigste bestehende Kategorie zur Plausibilisierung
        if doc_count > 0:
            category_counts = (
                source_df.iloc[doc_indices]["category_parent"]
                .value_counts()
                .head(3)
            )
            top_categories = " | ".join(
                [f"{cat}: {count}" for cat, count in category_counts.items()]
            )

            # Erste zugeordnete Meldungen auswählen und kürzen.
            examples = " || ".join(
                source_df.iloc[doc_indices][TEXT_COLUMN]
                .astype(str)
                .head(n_examples)
                .str.slice(0, 180)
                .tolist()
            )
        else:
            top_categories = ""
            examples = ""

        rows.append({
            "Topic": topic_number,
            "Anzahl Meldungen": doc_count,
            "Anteil Prozent": doc_share,
            "Bezugsmenge Meldungen": n_docs,
            "Top Begriffe": top_words,
            "Häufigste Kategorie": top_categories,
            "Beispielmeldungen": examples
        })

    summary = pd.DataFrame(rows)
    summary = summary.sort_values("Anzahl Meldungen", ascending=False).reset_index(drop=True)
    summary.insert(0, "Rang", np.arange(1, len(summary) + 1))

    if int(summary["Anzahl Meldungen"].sum()) != int(assignments["Topic"].notna().sum()):
        raise RuntimeError("Die Themenhäufigkeiten stimmen nicht mit den Zuordnungen überein.")
    return summary

lda_summary = build_topic_summary(
    lda_model,
    count_vectorizer.get_feature_names_out(),
    lda_assignments,
    df_model
)

nmf_summary = build_topic_summary(
    nmf_model,
    tfidf_vectorizer.get_feature_names_out(),
    nmf_assignments,
    df_model
)

print("LDA Themenübersicht:")
display(lda_summary)

print("NMF Themenübersicht:")
display(nmf_summary)

print("Die Anteile beziehen sich auf alle", N_ANALYSIS_DOCS, "Meldungen nach Text Preprocessing.")
print("Nicht zugeordnete Meldungen sind in der separaten Zuordnungsübersicht enthalten.")
print("Gerundete Prozentanteile können sich geringfügig von 100 Prozent unterscheiden.")


In [ ]:
# Schritt 14 Technische Themenübersichten als CSV speichern



# Die Spalte mit Rohtextbeispielen nicht in diesen CSV-Export übernehmen.
# Beispielmeldungen können Adressen oder personenbezogene Hinweise enthalten.
lda_export = lda_summary.drop(columns=["Beispielmeldungen"], errors="ignore")
nmf_export = nmf_summary.drop(columns=["Beispielmeldungen"], errors="ignore")

lda_path = OUTPUT_DIR / "themenuebersicht_lda.csv"
nmf_path = OUTPUT_DIR / "themenuebersicht_nmf.csv"

lda_export.to_csv(lda_path, index=False, encoding="utf-8-sig")
nmf_export.to_csv(nmf_path, index=False, encoding="utf-8-sig")

print("Gespeichert:")
print(lda_path.resolve())
print(nmf_path.resolve())

assignment_path = OUTPUT_DIR / "zuordnung_lda_nmf.csv"
assignment_summary.to_csv(assignment_path, index=False, sep=";", encoding="utf-8-sig")
print(assignment_path.resolve())


In [ ]:
# Schritt 15 NMF-Ergebnisse mit Themenbeschreibungen ergänzen

# Die bisherigen Bezeichnungen sind für das geänderte Modell noch nicht geprüft.
decision_table = None
if TOPIC_LABELS_REVIEWED:

    from sklearn.feature_extraction.text import CountVectorizer
    from pathlib import Path
    import numpy as np
    import pandas as pd

    # Bereits berechnete NMF-Zuordnungen übernehmen
    df_topic_docs = df_model.copy()
    df_topic_docs["NMF_Topic"] = nmf_assignments["Topic"]

    # Manuell formulierte Themenbezeichnungen, Beschreibungen und beispielhafte Hinweise.
    # Die Zuordnung gilt nur für den bisherigen Modellstand mit zehn Topics.
    # Nach erneutem Training ist die inhaltliche Zuordnung der Labels zu prüfen.
    # Die beispielhaften Hinweise sind keine wörtlichen Originalmeldungen.
    topic_meta = {
        1: {
            "Beschwerdethema": "Defekte Beleuchtung",
            "Kurzbeschreibung": "Viele Meldungen betreffen ausgefallene, flackernde oder schlecht sichtbare Beleuchtung im öffentlichen Raum.",
            "Typische Hinweise": "Laterne ausgefallen, Licht flackert, dunkler Bereich"
        },
        2: {
            "Beschwerdethema": "Herrenlose Fahrräder",
            "Kurzbeschreibung": "Viele Meldungen betreffen Fahrräder, die längere Zeit unbewegt stehen, beschädigt sind oder öffentliche Flächen blockieren.",
            "Typische Hinweise": "Fahrrad steht seit Wochen, Sattel fehlt, Fahrrad blockiert Platz"
        },
        3: {
            "Beschwerdethema": "Müll und Sauberkeit",
            "Kurzbeschreibung": "Viele Hinweise betreffen illegal abgelegten Müll, überfüllte Mülleimer und Verschmutzungen im öffentlichen Raum.",
            "Typische Hinweise": "Müll abgeladen, Mülleimer voll, Abfall im Gebüsch"
        },
        4: {
            "Beschwerdethema": "Gehwege und Heckenwuchs",
            "Kurzbeschreibung": "Meldungen betreffen zugewachsene Gehwege, Hecken, Engstellen und Einschränkungen für Fußgänger.",
            "Typische Hinweise": "Hecke wächst in Gehweg, Gehweg zu eng, Weg zugewachsen"
        },
        5: {
            "Beschwerdethema": "Straßenraum, Fahrzeuge und Behinderungen",
            "Kurzbeschreibung": "Das Thema bündelt Meldungen zu Straßen, Einfahrten, Parkplätzen, abgestellten Fahrzeugen und Behinderungen im Verkehrsraum.",
            "Typische Hinweise": "Fahrzeug abgestellt, Straße blockiert, Einfahrt beeinträchtigt"
        },
        6: {
            "Beschwerdethema": "Beschilderung",
            "Kurzbeschreibung": "Dieses Thema umfasst beschädigte, verschmutzte, umgefahrene oder schlecht erkennbare Schilder.",
            "Typische Hinweise": "Schild umgestürzt, Verkehrsschild verschmutzt, Schild schlecht sichtbar"
        },
        7: {
            "Beschwerdethema": "Radwege, Entwässerung und Gefahrenstellen",
            "Kurzbeschreibung": "Mehrere Meldungen betreffen Radwege und Wege mit Wasserstellen, verstopften Abläufen, Scherben oder anderen Gefahrenstellen.",
            "Typische Hinweise": "Wasser auf Radweg, Gulli verstopft, Scherben auf Weg"
        },
        8: {
            "Beschwerdethema": "Bäume, Äste und sonstige Hindernisse",
            "Kurzbeschreibung": "Meldungen beschreiben umgestürzte Bäume, überhängende Äste, Bewuchs oder Gegenstände, die Wege und Flächen blockieren.",
            "Typische Hinweise": "Baum umgestürzt, Äste hängen tief, Weg blockiert"
        },
        9: {
            "Beschwerdethema": "Ampeln und Fußgängerquerungen",
            "Kurzbeschreibung": "Meldungen beziehen sich auf Ampeln, Rot- und Grünphasen sowie Konflikte zwischen Fußgängern, Radfahrern und Verkehr.",
            "Typische Hinweise": "Ampel defekt, Rotphase zu lang, Querung unsicher"
        },
        10: {
            "Beschwerdethema": "Glasscherben und Verschmutzung an öffentlichen Orten",
            "Kurzbeschreibung": "Viele Meldungen betreffen Scherben und Verschmutzungen auf Spielplätzen, Radwegen, an Bushaltestellen oder auf öffentlichen Flächen.",
            "Typische Hinweise": "Glasscherben auf Spielplatz, Scherben auf Radweg, Müll an Haltestelle"
        }
    }

    # Prüfen, ob für jede Topic-Nummer eine Themenbezeichnung hinterlegt ist.
    # Diese Prüfung bewertet nicht, ob die Bezeichnungen inhaltlich passend sind.
    model_topics = set(nmf_summary["Topic"].astype(int))
    label_topics = set(topic_meta.keys())

    missing_labels = sorted(model_topics - label_topics)
    unused_labels = sorted(label_topics - model_topics)

    if missing_labels:
        raise ValueError(f"Für diese Topics fehlen Labels: {missing_labels}")

    if unused_labels:
        print(f"Hinweis: Diese Labels werden aktuell nicht genutzt: {unused_labels}")


    def top_child_categories(series, n=3):
        counts = series.value_counts().head(n)
        return "; ".join([f"{category} ({count})" for category, count in counts.items()])


    # Kombinationen aus bereinigten Tokens sind nicht zwangsläufig Originalphrasen.
    def top_ngrams_for_topic(texts, top_n=5):
        texts = [text for text in texts if isinstance(text, str) and len(text.split()) >= 2]

        if len(texts) == 0:
            return ""

        min_df_value = 2 if len(texts) >= 5 else 1

        vectorizer = CountVectorizer(
            ngram_range=(2, 3),
            min_df=min_df_value,
            token_pattern=r"(?u)\b[a-zäöüß][a-zäöüß]+\b"
        )

        try:
            X = vectorizer.fit_transform(texts)
        except ValueError:
            return ""

        terms = vectorizer.get_feature_names_out()
        scores = np.asarray(X.sum(axis=0)).ravel()

        top_indices = scores.argsort()[::-1][:top_n]
        return "; ".join([terms[i] for i in top_indices])


    # Dominante Unterkategorien je Topic berechnen
    child_summary = (
        df_topic_docs
        .groupby("NMF_Topic")["category_child"]
        .apply(top_child_categories)
        .reset_index()
        .rename(columns={
            "NMF_Topic": "Topic",
            "category_child": "Dominante Unterkategorien"
        })
    )

    # Häufige Kombinationen aus zwei oder drei Tokens im bereinigten Text zählen.
    phrase_rows = []

    for topic_id in sorted(model_topics):
        topic_texts = df_topic_docs.loc[
            df_topic_docs["NMF_Topic"] == topic_id,
            "text_clean"
        ].tolist()

        phrase_rows.append({
            "Topic": topic_id,
            "Auffällige Wortkombinationen": top_ngrams_for_topic(topic_texts, top_n=5)
        })

    phrase_summary = pd.DataFrame(phrase_rows)

    # Manuell hinterlegte Themenangaben in einen DataFrame überführen.
    topic_meta_df = (
        pd.DataFrame.from_dict(topic_meta, orient="index")
        .rename_axis("Topic")
        .reset_index()
    )

    # Entscheidungsträger-Tabelle bauen
    decision_table = (
        nmf_summary[["Topic", "Anzahl Meldungen", "Anteil Prozent"]]
        .merge(topic_meta_df, on="Topic", how="left")
        .merge(child_summary, on="Topic", how="left")
        .merge(phrase_summary, on="Topic", how="left")
    )

    decision_table = decision_table.sort_values(
        "Anzahl Meldungen",
        ascending=False
    ).reset_index(drop=True)

    decision_table.insert(0, "Rang", np.arange(1, len(decision_table) + 1))

    decision_table["Anteil"] = decision_table["Anteil Prozent"].map(
        lambda value: f"{value:.1f} %"
    )

    decision_table = decision_table[
        [
            "Rang",
            "Beschwerdethema",
            "Kurzbeschreibung",
            "Anzahl Meldungen",
            "Anteil",
            "Dominante Unterkategorien",
            "Typische Hinweise",
            "Auffällige Wortkombinationen"
        ]
    ]

    display(decision_table)
else:
    print("Beschriftete Ergebnisübersicht und Diagramm bleiben bis zur Prüfung der Themenbezeichnungen deaktiviert.")


In [ ]:
# Schritt 16 Übersicht für Entscheidungstragende aufbereiten

decision_table_final = None
if decision_table is not None:

    decision_table_final = decision_table.copy()

    # Spalten für die Übersicht für Entscheidungstragende auswählen.
    # Die zur Plausibilisierung verwendeten Kategorien bleiben in der technischen Übersicht.
    decision_table_final = decision_table_final[
        [
            "Rang",
            "Beschwerdethema",
            "Kurzbeschreibung",
            "Anzahl Meldungen",
            "Anteil",
            "Typische Hinweise"
        ]
    ].copy()

    # Ausgewählte Themenbezeichnungen an die bisherige inhaltliche Interpretation anpassen.
    decision_table_final["Beschwerdethema"] = decision_table_final["Beschwerdethema"].replace({
        "Radwege, Entwässerung und Gefahrenstellen": "Gefahrenstellen auf Wegen und Radwegen",
        "Bäume, Äste und sonstige Hindernisse": "Hindernisse im öffentlichen Raum",
        "Glasscherben und Verschmutzung an öffentlichen Orten": "Scherben und Verschmutzung an öffentlichen Orten"
    })

    # Kurzbeschreibungen an die finalen Labels anpassen
    decision_table_final.loc[
        decision_table_final["Beschwerdethema"] == "Gefahrenstellen auf Wegen und Radwegen",
        "Kurzbeschreibung"
    ] = "Mehrere Meldungen betreffen Gefahrenstellen auf Wegen und Radwegen, darunter Scherben, Wasser, verstopfte Abläufe oder Hindernisse."

    decision_table_final.loc[
        decision_table_final["Beschwerdethema"] == "Hindernisse im öffentlichen Raum",
        "Kurzbeschreibung"
    ] = "Meldungen beschreiben blockierte oder eingeschränkte öffentliche Flächen, zum Beispiel durch Bäume, Äste, abgestellte Gegenstände oder andere Hindernisse."

    decision_table_final.loc[
        decision_table_final["Beschwerdethema"] == "Scherben und Verschmutzung an öffentlichen Orten",
        "Kurzbeschreibung"
    ] = "Viele Meldungen betreffen Scherben und Verschmutzungen auf Spielplätzen, Radwegen, an Bushaltestellen oder auf öffentlichen Flächen."

    # Textfelder bereinigen
    decision_table_final["Typische Hinweise"] = decision_table_final["Typische Hinweise"].astype(str).str.strip()
    decision_table_final["Anteil"] = decision_table_final["Anteil"].astype(str).str.strip()

    # Spaltenname präzisieren
    decision_table_final = decision_table_final.rename(
        columns={"Typische Hinweise": "Beispielhafte Hinweise"}
    )

    display(decision_table_final)

if decision_table_final is not None:
    print("Bezugsmenge:", N_ANALYSIS_DOCS, "Meldungen nach Text Preprocessing.")
    print("Nicht zugeordnet:", int(nmf_assignments["Topic"].isna().sum()))


In [ ]:
# Schritt 17 Übersicht für Entscheidungstragende speichern

if decision_table_final is not None:

    decision_table_final_path = OUTPUT_DIR / "entscheidungstraeger_themenuebersicht_final.csv"

    decision_table_final.to_csv(
        decision_table_final_path,
        index=False,
        sep=";",
        encoding="utf-8-sig"
    )

    print("Finale Entscheidungsträger-Übersicht gespeichert:")
    print(decision_table_final_path.resolve())


In [ ]:
# Schritt 18 Themenhäufigkeiten als Balkendiagramm darstellen

if decision_table_final is not None:

    import matplotlib.pyplot as plt

    chart_df = decision_table_final.sort_values("Anzahl Meldungen", ascending=True)

    plt.figure(figsize=(10, 6))
    plt.barh(chart_df["Beschwerdethema"], chart_df["Anzahl Meldungen"])
    plt.xlabel("Anzahl Meldungen")
    plt.ylabel("Beschwerdethema")
    plt.title("Häufigste Beschwerdethemen im Mängelmelder Konstanz")
    plt.tight_layout()

    chart_path = OUTPUT_DIR / "haeufigste_beschwerdethemen_final.png"
    plt.savefig(chart_path, dpi=150)
    plt.show()

    print("Diagramm gespeichert:")
    print(chart_path.resolve())
